# Music Popularity, Genre Patterns and a Tag-Based Recommender

**Course:** Data Collection & Preparation — SIS №1

**Goal.** Collect real music data from two different sources, clean and merge it, explore how
chart success relates to long-term listening popularity and genres, and build a simple
content-based recommender from user-assigned tags.

| Source | Type | What we take |
|---|---|---|
| Wikipedia — *Billboard Year-End Hot 100 singles of &lt;year&gt;* (2010–2025) | static web pages, `requests` + `BeautifulSoup` | year, chart position, title, artist(s) |
| Last.fm API (`track.getInfo`, `artist.getTopTags`) | open REST API (JSON) | listeners, play count, duration, user tags |

**Pipeline:** scrape → call API → clean → merge → EDA → visualizations → recommender.

In [ ]:
import json, os, re, time
from pathlib import Path
from urllib.robotparser import RobotFileParser

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

YEARS = range(2010, 2026)
HEADERS = {"User-Agent": "KBTU-SIS-student-project/1.0 (educational use)"}
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. Web scraping — Billboard Year-End Hot 100 (Wikipedia)

First we check `robots.txt` to make sure the pages we need are allowed for crawling,
then download one page per year (with a pause between requests) and parse the chart table.

In [ ]:
WIKI = "https://en.wikipedia.org"
page_url = lambda year: f"{WIKI}/wiki/Billboard_Year-End_Hot_100_singles_of_{year}"

robots = RobotFileParser()
robots.parse(requests.get(f"{WIKI}/robots.txt", headers=HEADERS, timeout=30).text.splitlines())
allowed = all(robots.can_fetch(HEADERS["User-Agent"], page_url(y)) for y in YEARS)
print("robots.txt allows scraping these pages:", allowed)
assert allowed

In [ ]:
def scrape_year(year):
    # Return the year-end chart of one year as a list of [year, no, title, artist] rows.
    resp = requests.get(page_url(year), headers=HEADERS, timeout=30)
    resp.raise_for_status()
    table = BeautifulSoup(resp.text, "html.parser").find("table", class_="wikitable")

    rows, carry = [], {}  # carry: column index -> [text, rows left] for cells with rowspan
    for tr in table.find_all("tr")[1:]:
        cells, row = iter(tr.find_all(["th", "td"])), []
        for col in range(3):
            if carry.get(col, [None, 0])[1] > 0:  # cell is covered by a rowspan from above
                row.append(carry[col][0])
                carry[col][1] -= 1
                continue
            cell = next(cells)
            text = " ".join(cell.get_text().split())
            span = int(cell.get("rowspan", 1))
            if span > 1:
                carry[col] = [text, span - 1]
            row.append(text)
        rows.append([year, *row])
    return rows


scraped = []
for year in YEARS:
    rows = scrape_year(year)
    assert len(rows) == 100, f"{year}: expected 100 rows, got {len(rows)}"
    scraped += rows
    time.sleep(1)  # be polite to Wikipedia

chart_raw = pd.DataFrame(scraped, columns=["year", "No.", "Title", "Artist(s)"])
chart_raw.to_csv(DATA / "billboard_raw.csv", index=False)
print(chart_raw.shape)
chart_raw.head()

### Cleaning the scraped data

Raw titles are wrapped in quotes and may contain footnote marks like `[a]`; the artist field mixes
the main artist with featured guests. We rename columns, fix data types and derive a
`main_artist` (without "featuring ...") and a `primary_artist` (first credited name) that are used to query the API.

In [ ]:
def strip_notes(s):
    return re.sub(r"\[.*?\]", "", s).strip()

chart = chart_raw.rename(columns={"No.": "rank", "Title": "title", "Artist(s)": "artist"})
chart["rank"] = chart["rank"].map(strip_notes).astype(int)
chart["title"] = chart["title"].map(strip_notes).str.replace('"', "").str.strip()
chart["artist"] = chart["artist"].map(strip_notes)
chart["has_feature"] = chart["artist"].str.contains(r"\bfeaturing\b", case=False)
chart["main_artist"] = chart["artist"].str.split(r"(?i)\s+featuring\s+", regex=True).str[0]
chart["primary_artist"] = chart["main_artist"].str.split(r"\s+(?:and|&|with|x)\s+|,\s+", regex=True).str[0]

print("missing values:\n", chart.isna().sum().to_string())
print("exact duplicate rows:", chart.duplicated().sum())
print("songs that appear in more than one year-end chart:",
      chart.duplicated(["title", "artist"]).sum())
chart.head()

## 2. API — Last.fm

For every unique scraped song we call `track.getInfo` (listeners, play count, duration, top tags).
If a track has no tags of its own we fall back to the artist's tags (`artist.getTopTags`).

* The API key is read from the `LASTFM_API_KEY` environment variable or a local `.env` file (not committed).
* Every response is cached in `data/lastfm_cache.json`, so re-running the notebook does not hit the API again
  (and the notebook can be reproduced without a key).

In [ ]:
API_URL = "https://ws.audioscrobbler.com/2.0/"
CACHE_FILE = DATA / "lastfm_cache.json"

if not os.getenv("LASTFM_API_KEY") and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if line.startswith("LASTFM_API_KEY="):
            os.environ["LASTFM_API_KEY"] = line.split("=", 1)[1].strip().strip('"')
API_KEY = os.getenv("LASTFM_API_KEY")

cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {"tracks": {}, "artist_tags": {}}
RETRYABLE = {-1, 8, 11, 16, 29}  # network problem, backend errors, rate limit


def lastfm(method, **params):
    # Call a Last.fm API method; returns parsed JSON ({'error': ...} on failure).
    if not API_KEY:
        raise RuntimeError("Set LASTFM_API_KEY (env variable or .env file) to fetch uncached data")
    params = {**params, "method": method, "api_key": API_KEY, "format": "json", "autocorrect": 1}
    for attempt in range(3):
        try:
            data = requests.get(API_URL, params=params, headers=HEADERS, timeout=30).json()
        except (requests.RequestException, ValueError):
            data = {"error": -1}
        if data.get("error") == 10:
            raise RuntimeError("Last.fm rejected the API key")
        if data.get("error") not in RETRYABLE:
            break
        time.sleep(2 * (attempt + 1))
    time.sleep(0.2)  # stay well under the rate limit
    return data


def tag_names(container):
    # 'toptags' is a dict with a list (or a single dict) under 'tag', or an empty string.
    tags = container.get("tag", []) if isinstance(container, dict) else []
    return [t["name"] for t in ([tags] if isinstance(tags, dict) else tags)]


def fetch_track(title, artists):
    # Try each artist spelling until Last.fm finds the track. None = temporary failure.
    for artist in dict.fromkeys(artists):
        data = lastfm("track.getInfo", track=title, artist=artist)
        if "track" in data:
            t = data["track"]
            return {"found": True, "lf_title": t["name"], "lf_artist": t["artist"]["name"],
                    "listeners": int(t["listeners"]), "playcount": int(t["playcount"]),
                    "duration_ms": int(t.get("duration") or 0), "track_tags": tag_names(t.get("toptags"))}
        if data.get("error") in RETRYABLE:
            return None
    return {"found": False}


def fetch_artist_tags(artist):
    data = lastfm("artist.getTopTags", artist=artist)
    if data.get("error") in RETRYABLE:
        return None
    return tag_names(data.get("toptags"))[:5]

In [ ]:
songs = chart.drop_duplicates(["title", "artist"])
for i, s in enumerate(songs.itertuples()):
    key = f"{s.artist}|||{s.title}"
    if key not in cache["tracks"]:
        rec = fetch_track(s.title, [s.main_artist, s.primary_artist])
        if rec is not None:
            cache["tracks"][key] = rec
    rec = cache["tracks"].get(key)
    if rec and rec["found"] and not rec["track_tags"] and rec["lf_artist"] not in cache["artist_tags"]:
        tags = fetch_artist_tags(rec["lf_artist"])
        if tags is not None:
            cache["artist_tags"][rec["lf_artist"]] = tags
    if i % 100 == 0:
        CACHE_FILE.write_text(json.dumps(cache))
CACHE_FILE.write_text(json.dumps(cache))

lastfm_raw = pd.DataFrame(
    [{"artist": k.split("|||")[0], "title": k.split("|||")[1], **v} for k, v in cache["tracks"].items()]
)
print(lastfm_raw.shape, "| found on Last.fm:", int(lastfm_raw["found"].sum()))
lastfm_raw.head()

### Cleaning the API data

* numeric columns → proper numeric types; a duration of `0` means "unknown" on Last.fm → `NaN`, then filled with the median;
* tags: fall back to artist tags, lower-case, unify spelling variants, drop non-genre tags (years, "favorites", the artist's own name, ...);
* tracks that Last.fm could not find are removed (reported below).

In [ ]:
TAG_ALIASES = {"hip hop": "hip-hop", "hiphop": "hip-hop", "r&b": "rnb", "r and b": "rnb",
               "rhythm and blues": "rnb", "electronica": "electronic", "electro": "electronic"}
TAG_STOP = {"favorites", "favourites", "favorite", "seen live", "love", "loved", "awesome", "catchy",
            "my top songs", "love at first listen", "female vocalists", "male vocalists",
            "female vocalist", "male vocalist", "american", "british", "canadian", "usa", "uk",
            "billboard hot 100", "number one", "best of 2010s", "under 2000 listeners"}


def clean_tags(row):
    raw = row["track_tags"] if isinstance(row["track_tags"], list) and row["track_tags"] else row["artist_tags"]
    out = []
    for tag in raw if isinstance(raw, list) else []:
        tag = TAG_ALIASES.get(tag.lower().strip(), tag.lower().strip())
        is_noise = (tag in TAG_STOP or re.fullmatch(r"[\d\s's-]+", tag)
                    or tag == str(row["lf_artist"]).lower())
        if not is_noise and tag not in out:
            out.append(tag)
    return out


lf = lastfm_raw[lastfm_raw["found"]].copy()
print("tracks not found on Last.fm (dropped):", int((~lastfm_raw["found"]).sum()))

lf["artist_tags"] = lf["lf_artist"].map(cache["artist_tags"])
lf["tag_source"] = np.where(lf["track_tags"].str.len() > 0, "track", "artist")
lf["tags"] = lf.apply(clean_tags, axis=1)
lf[["listeners", "playcount"]] = lf[["listeners", "playcount"]].astype("int64")
lf["duration_min"] = lf["duration_ms"].replace(0, np.nan) / 60000

print("missing duration:", int(lf["duration_min"].isna().sum()), "-> filled with median")
print("tracks without any tag:", int((lf["tags"].str.len() == 0).sum()))
lf["duration_min"] = lf["duration_min"].fillna(lf["duration_min"].median())

lf = lf.drop(columns=["found", "duration_ms", "track_tags", "artist_tags"])
lf.info()

## 3. Merging the two sources

The scraped chart (one row per *year × position*) is joined with the Last.fm data on the
`(title, artist)` pair. `validate="m:1"` guarantees that the API table has exactly one row per song.

In [ ]:
merged = chart.merge(lf, on=["title", "artist"], how="inner", validate="m:1")
merged["plays_per_listener"] = merged["playcount"] / merged["listeners"]
merged["log_playcount"] = np.log10(merged["playcount"].clip(lower=1))
print(f"chart entries: {len(chart)} -> merged: {len(merged)} ({len(merged) / len(chart):.1%} matched)")

# One row per song (a song can be in two year-end charts): keep its best position.
tracks = (merged.sort_values("rank")
          .drop_duplicates(["lf_title", "lf_artist"])
          .sort_values(["year", "rank"]).reset_index(drop=True))
print("unique tracks:", len(tracks))
merged.to_csv(DATA / "merged.csv", index=False)
merged.head()

## 4. Exploratory data analysis

### Descriptive statistics

In [ ]:
num_cols = ["rank", "listeners", "playcount", "plays_per_listener", "duration_min"]
tracks[num_cols].describe().T.round(2)

In [ ]:
print("Most played chart hits on Last.fm:")
display(tracks.nlargest(10, "playcount")[["year", "rank", "title", "artist", "listeners", "playcount"]])

print("Artists with the most year-end chart entries:")
display(merged["primary_artist"].value_counts().head(10).to_frame("chart_entries"))

### Correlations

Spearman is used because play counts are heavily right-skewed and `rank` is ordinal.

In [ ]:
corr = merged[["year", *num_cols]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Spearman correlation between chart and Last.fm metrics")
plt.tight_layout()
plt.show()
print("rank vs playcount, within each year (mean Spearman):",
      round(merged.groupby("year")[["rank", "playcount"]].corr(method="spearman").xs("rank", level=1)["playcount"].mean(), 3))

## 5. Visualizations

### 5.1 Popularity distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(tracks["playcount"] / 1e6, bins=40, ax=axes[0])
axes[0].set(title="Play counts are heavily right-skewed", xlabel="Last.fm play count (millions)", ylabel="Number of tracks")
sns.histplot(tracks["log_playcount"], bins=40, kde=True, ax=axes[1])
axes[1].set(title="...and roughly bell-shaped on a log scale", xlabel="log10(play count)", ylabel="Number of tracks")
plt.tight_layout()
plt.show()

### 5.2 Chart position vs. long-term popularity

In [ ]:
merged["rank_group"] = pd.cut(merged["rank"], [0, 10, 25, 50, 75, 100],
                              labels=["1-10", "11-25", "26-50", "51-75", "76-100"])
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=merged, x="rank_group", y="playcount", color="tab:blue", ax=ax)
ax.set(yscale="log", title="Higher year-end positions get more Last.fm plays",
       xlabel="Billboard year-end position", ylabel="Last.fm play count (log scale)")
plt.tight_layout()
plt.show()

### 5.3 Most common genres / tags

In [ ]:
tag_rows = merged.explode("tags").dropna(subset=["tags"])
top_tags = tag_rows.drop_duplicates(["lf_title", "lf_artist", "tags"])["tags"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=top_tags.values, y=top_tags.index, color="tab:blue", ax=ax)
ax.set(title="Top 15 Last.fm tags among Billboard year-end hits (2010-2025)",
       xlabel="Number of tracks", ylabel="")
plt.tight_layout()
plt.show()

### 5.4 Genre trends over the years

In [ ]:
GENRES = ["pop", "hip-hop", "rap", "rnb", "country", "rock", "dance", "electronic", "trap", "indie"]
trend_genres = [g for g in top_tags.index if g in GENRES][:6]
share = (tag_rows[tag_rows["tags"].isin(trend_genres)]
         .groupby(["year", "tags"]).size().unstack(fill_value=0)
         .div(merged.groupby("year").size(), axis=0) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
share[trend_genres].plot(marker="o", ax=ax)
ax.set(title="Share of year-end chart entries carrying each genre tag",
       xlabel="Chart year", ylabel="% of chart entries")
ax.legend(title="Tag", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 5.5 Song duration and collaborations over time

In [ ]:
by_year = merged.groupby("year").agg(duration_min=("duration_min", "median"),
                                      feature_share=("has_feature", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
by_year["duration_min"].plot(marker="o", ax=axes[0])
axes[0].set(title="Median song duration", xlabel="Chart year", ylabel="Minutes")
(by_year["feature_share"] * 100).plot(marker="o", color="tab:orange", ax=axes[1])
axes[1].set(title='Share of hits with a "featuring" credit', xlabel="Chart year", ylabel="% of chart entries")
plt.tight_layout()
plt.show()

## 6. Content-based recommender (tag similarity)

Each track is described by its tags. Tags are turned into TF-IDF vectors (rare, specific tags
such as *k-pop* weigh more than ubiquitous ones such as *pop*) and tracks are compared with
cosine similarity. For a seed song, the recommender returns the most similar other tracks;
ties are broken by Last.fm listeners.

In [ ]:
rec_df = tracks[tracks["tags"].str.len() > 0].reset_index(drop=True)
vectorizer = TfidfVectorizer(analyzer=lambda tags: tags)
tfidf = vectorizer.fit_transform(rec_df["tags"])
similarity = cosine_similarity(tfidf)
print(f"{tfidf.shape[0]} tracks x {tfidf.shape[1]} distinct tags")


def recommend(title, n=10):
    # Top-n tracks most similar to `title` by tag profile (other artists only).
    matches = rec_df.index[rec_df["lf_title"].str.lower() == title.lower()]
    if len(matches) == 0:
        raise ValueError(f"'{title}' is not in the dataset")
    seed = matches[0]
    out = rec_df.assign(similarity=similarity[seed])
    out = out[out["lf_artist"] != rec_df.loc[seed, "lf_artist"]]
    return (out.sort_values(["similarity", "listeners"], ascending=False)
            .head(n)[["lf_title", "lf_artist", "year", "tags", "similarity"]])


# self-check: results are sorted, exclude the seed artist, and similarity is a valid cosine
_seed = rec_df.loc[0]
_r = recommend(_seed["lf_title"], 5)
assert _r["similarity"].is_monotonic_decreasing and _r["similarity"].between(0, 1 + 1e-9).all()
assert _seed["lf_artist"] not in set(_r["lf_artist"])

In [ ]:
SEEDS = ["Blinding Lights", "Thrift Shop", "Radioactive"]
for seed in SEEDS:
    seed_row = rec_df[rec_df["lf_title"].str.lower() == seed.lower()].iloc[0]
    print(f"\nBecause you like '{seed_row.lf_title}' by {seed_row.lf_artist}  (tags: {', '.join(seed_row.tags)})")
    display(recommend(seed, 5).round({"similarity": 3}))

In [ ]:
fig, axes = plt.subplots(1, len(SEEDS), figsize=(16, 4.5))
for ax, seed in zip(axes, SEEDS):
    recs = recommend(seed, 8)
    sns.barplot(x=recs["similarity"], y=recs["lf_title"] + " — " + recs["lf_artist"], color="tab:green", ax=ax)
    ax.set(title=f"Similar to '{seed}'", xlabel="Cosine similarity of tags", ylabel="", xlim=(0, 1))
plt.tight_layout()
plt.show()

### Tag similarity between the most frequent artists

In [ ]:
top_artists = rec_df["lf_artist"].value_counts().head(15).index
artist_docs = rec_df[rec_df["lf_artist"].isin(top_artists)].groupby("lf_artist")["tags"].sum()
artist_sim = pd.DataFrame(cosine_similarity(vectorizer.transform(artist_docs)),
                          index=artist_docs.index, columns=artist_docs.index)

grid = sns.clustermap(artist_sim, cmap="viridis", figsize=(8.5, 8.5), vmin=0, vmax=1,
                      cbar_kws={"label": "Cosine similarity"})
grid.fig.suptitle("Artists with the most chart hits, clustered by tag similarity", y=1.02)
grid.ax_heatmap.set(xlabel="", ylabel="")
plt.show()

## 7. Conclusions

*(to be filled in after the full run with real numbers)*